## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [ ]:
import gradio as gr
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage

# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
MODEL = "gemini-3.7-flash"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [6]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [7]:
retriever.invoke("Who is Avery?")

[Document(id='bae75b87-374d-4f2d-9295-75907725e2e9', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [8]:
# well this will give a generic answer on the Avery first name... it's still
# missing all info from the retriever!
llm.invoke("Who is Avery?")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Time to put this together!

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt),
                           HumanMessage(content=question)])
    return response.content

In [11]:
# Aaaaan now it's correct!
answer_question("Who is Averi Lancaster?", [])

[{'type': 'text',
  'text': 'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. \n\nHere are a few key details about her:\n* **Role at Insurellm:** She co-founded the company in 2015 and has served as CEO since, guiding Insurellm to become a leading Insurance Tech provider through her innovative leadership strategies and risk management expertise.\n* **Previous Experience:** Prior to founding Insurellm, she was a Senior Product Manager at Innovate Insurance Solutions from 2013 to 2015.\n* **Location:** San Francisco, California.',
  'extras': {'signature': 'ErIJCq8JARFNMg/8/2zh+6HprvRfR2pzqFPV7XUse2k51apx7XKc/HgDKigEvh+4VN5K5NipgVsCF52BHxw+MXKjX83amRct9T6JBzv+VE8r97UNmWp7GLry3dZXVX1/cLFSJ1JhwbpPwg9J49cGGI8w3a8Ug0kCy4o5pU+nLok4IjQVBUN5rLBopVyI7JNvI/bDn/FaFB1duKgdjW/noYwvzLKZiqh0aql3MWE+WSLoLaE9Gb8mS8koVdkj4aEDNTcIiXWvSp0Gfz5VMPz679ztPfUdcYfK1bEc8pN43EO4Gb/NV2FMZZNk2LLgpk4IUdNGqX9qIGZENYshyrQ9hPKkyrEhdoQj3aGh/CR3gmTzOWsccTq+lBfUnOjXoNtSxvGn5CV9Zc5h1SO6mVqs6

## What could possibly come next? 😂

In [12]:
gr.ChatInterface(answer_question).launch()

c:\Users\Pazzucconibt\REPO\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!